# 00 — Présentation du jeu de données Olist

**Objectif du projet :** construire, à partir des 9 fichiers CSV, un DataFrame unique
**complet** (aucune valeur manquante) et **cohérent**, sans supprimer de lignes.
Un modèle de Machine Learning sera entraîné dessus plus tard (cible non encore choisie).

**Source :** *Brazilian E-Commerce Public Dataset by Olist*, publié sur Kaggle
(`kaggle.com/datasets/olistbr/brazilian-ecommerce`). Commandes passées sur la place de
marché brésilienne Olist entre septembre 2016 et octobre 2018.

Ce notebook décrit le jeu de données dans son ensemble. Le nettoyage de chaque table
est fait dans son propre notebook (voir le plan en fin de notebook).

## Chargement des fichiers

In [1]:
import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

# un DataFrame par fichier, nommé sans le préfixe "olist_" ni le suffixe "_dataset"
df_dic = {}
for file in sorted(glob.glob("*.csv")):
    nom = file.replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## Les 9 tables

| Table | Contenu |
|---|---|
| `orders` | les commandes : statut et dates du parcours (achat, approbation, remise au transporteur, livraison, livraison estimée) |
| `customers` | le client de chaque commande et sa localisation |
| `order_items` | les articles de chaque commande : produit, vendeur, prix, frais de port |
| `order_payments` | les paiements de chaque commande : moyen, nombre de mensualités, montant |
| `order_reviews` | les avis laissés sur les commandes : note de 1 à 5, titre, commentaire |
| `products` | les produits : catégorie (en portugais), longueur du nom et de la description, photos, poids et dimensions |
| `sellers` | les vendeurs et leur localisation |
| `product_category_name_translation` | la traduction anglaise des catégories de produits |
| `geolocation` | les coordonnées GPS associées aux codes postaux |

In [2]:
# vue d'ensemble : taille, doublons exacts et valeurs manquantes de chaque table
pd.DataFrame(
    {
        "lignes": {nom: len(df) for nom, df in df_dic.items()},
        "colonnes": {nom: df.shape[1] for nom, df in df_dic.items()},
        "doublons exacts": {nom: df.duplicated().sum() for nom, df in df_dic.items()},
        "valeurs manquantes": {nom: df.isna().sum().sum() for nom, df in df_dic.items()},
    }
)

,lignes,colonnes,doublons exacts,valeurs manquantes
customers,99441,5,0,0
geolocation,1000163,5,261831,0
order_items,112650,7,0,0
order_payments,103886,5,0,0
order_reviews,99224,7,0,145903
orders,99441,8,0,4908
products,32951,9,0,2448
sellers,3095,4,0,0
product_category_name_translation,71,2,0,0


## Aperçu détaillé de chaque table

In [3]:
for nom, df in df_dic.items():
    print("=====", nom, "=====")
    display(resume(df))

===== customers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
customer_id,str,99441,ffffe8b65bbe3087b653a978c870db99,00012a2ce6f8dcda20d059ce98491703,0,0.0
customer_unique_id,str,96096,ffffd2657e2aad2907e67c3e9daecbeb,0000366f3b9a7992bf8c76cfdf3221e2,0,0.0
customer_zip_code_prefix,int64,14994,99990,1003,0,0.0
customer_city,str,4119,zortea,abadia dos dourados,0,0.0
customer_state,str,27,TO,AC,0,0.0


===== geolocation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
geolocation_zip_code_prefix,int64,19015,99990,1001,0,0.0
geolocation_lat,float64,717360,45.065933,-36.605374,0,0.0
geolocation_lng,float64,717613,121.105394,-101.466766,0,0.0
geolocation_city,str,8011,óleo,* cidade,0,0.0
geolocation_state,str,27,TO,AC,0,0.0


===== order_items =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
order_id,str,98666,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.0
order_item_id,int64,21,21,1,0,0.0
product_id,str,32951,fffe9eeff12fcbd74a2f2b007dde0c58,00066f42aeeb9f3007548bb9d3f33c38,0,0.0
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
shipping_limit_date,str,93318,2020-04-09 22:35:08,2016-09-19 00:15:34,0,0.0
price,float64,5968,6735.0,0.85,0,0.0
freight_value,float64,6999,409.68,0.0,0,0.0


... (8 outputs left out)

===== sellers =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
seller_id,str,3095,ffff564a4f9085cd26170f4732393726,0015a82c2db000af6aaaf3ae2ecb0532,0,0.0
seller_zip_code_prefix,int64,2246,99730,1001,0,0.0
seller_city,str,611,xaxim,04482255,0,0.0
seller_state,str,23,SP,AC,0,0.0


===== product_category_name_translation =====


,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
product_category_name,str,71,utilidades_domesticas,agro_industria_e_comercio,0,0.0
product_category_name_english,str,71,watches_gifts,agro_industry_and_commerce,0,0.0


## Niveau de détail des tables

Le **niveau de détail** d'une table, c'est ce que représente une de ses lignes. Il
décide de la façon de fusionner : si on fusionne deux tables qui ont chacune
plusieurs lignes par commande, les lignes se multiplient entre elles (3 articles
× 2 paiements = 6 lignes) et les sommes deviennent fausses.

**Choix du groupe :** le DataFrame final aura **une ligne par article commandé**
(niveau de détail de `order_items`).

In [4]:
# nombre maximal de lignes par valeur de clé :
# 1 = une seule ligne par clé ; plus de 1 = plusieurs lignes à agréger avant de fusionner
cles = {
    "orders": "order_id",
    "customers": "customer_id",
    "order_items": "order_id",
    "order_payments": "order_id",
    "order_reviews": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
    "product_category_name_translation": "product_category_name",
    "geolocation": "geolocation_zip_code_prefix",
}

pd.DataFrame(
    {
        "clé": cles,
        "lignes max par clé": {
            nom: df_dic[nom].groupby(cle).size().max() for nom, cle in cles.items()
        },
    }
)

,clé,lignes max par clé
orders,order_id,1
customers,customer_id,1
order_items,order_id,21
order_payments,order_id,29
order_reviews,order_id,3
products,product_id,1
sellers,seller_id,1
product_category_name_translation,product_category_name,1
geolocation,geolocation_zip_code_prefix,1146


## Liens entre les tables

```
customers ──customer_id── orders ──order_id──┬── order_items ──product_id── products ──product_category_name── translation
    │                                        │        │
    │ zip_code_prefix                        │        └──seller_id── sellers
    ▼                                        ├── order_payments           │ zip_code_prefix
geolocation ◄────────────────────────────────┼────────────────────────────┘
                                             └── order_reviews
```

La cellule suivante vérifie, pour chaque lien, que les valeurs de la clé existent bien
dans la table visée. Une valeur absente donnera une valeur manquante après la fusion.

In [5]:
# (table source, colonne source, table visée, colonne visée)
liens = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("products", "product_category_name", "product_category_name_translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]

controle = []
for source, col, cible, col_cible in liens:
    valeurs = df_dic[source][col].dropna()  # une valeur manquante n'est pas un lien cassé
    absentes = ~valeurs.isin(df_dic[cible][col_cible])
    controle.append(
        {
            "lien": f"{source}.{col} → {cible}",
            "lignes sans correspondance": absentes.sum(),
            "valeurs distinctes sans correspondance": valeurs[absentes].nunique(),
        }
    )

pd.DataFrame(controle)

,lien,lignes sans correspondance,valeurs distinctes sans correspondance
0,orders.customer_id → customers,0,0
1,order_items.order_id → orders,0,0
2,order_payments.order_id → orders,0,0
3,order_reviews.order_id → orders,0,0
4,order_items.product_id → products,0,0
5,order_items.seller_id → sellers,0,0
6,products.product_category_name → product_categ...,13,2
7,customers.customer_zip_code_prefix → geolocation,278,157
8,sellers.seller_zip_code_prefix → geolocation,7,7


In [6]:
# dans l'autre sens : commandes qui n'apparaissent pas dans les tables qui en dépendent
commandes = df_dic["orders"]["order_id"]
pd.Series(
    {
        "commandes sans article": (~commandes.isin(df_dic["order_items"]["order_id"])).sum(),
        "commandes sans paiement": (~commandes.isin(df_dic["order_payments"]["order_id"])).sum(),
        "commandes sans avis": (~commandes.isin(df_dic["order_reviews"]["order_id"])).sum(),
    }
)

commandes sans article     775
commandes sans paiement      1
commandes sans avis        768
dtype: int64

## Conséquences pour le nettoyage et la fusion

- **Tables à plusieurs lignes par commande** (`order_payments`, `order_reviews`) :
  à agréger à une ligne par commande avant de les joindre aux articles.
- **`geolocation`** : plusieurs centaines de points GPS par code postal, et de nombreux
  doublons exacts. À réduire à une ligne par code postal.
- **Codes postaux absents de `geolocation`** (clients et vendeurs) et **catégories sans
  traduction** : ces liens produiront des valeurs manquantes à la fusion, à combler
  pour respecter la consigne.
- **Commandes sans article** : elles n'apparaîtront pas dans un DataFrame construit à
  partir de `order_items` (une ligne par article). Ce n'est pas une suppression, car ces
  commandes n'ont aucun article à représenter, mais c'est à signaler dans le notebook de fusion.
- **`customer_id` change à chaque commande** : l'identifiant d'une personne est
  `customer_unique_id`.

## Décisions du groupe

- Aucune ligne supprimée ; aucune valeur manquante dans le DataFrame final.
- Les valeurs inventées ou corrigées restent repérables grâce à des colonnes booléennes
  `imputee_none` (valeur manquante remplacée) et `imputee_error` (valeur incohérente corrigée).

## Plan des notebooks

| Notebook | Rôle |
|---|---|
| `00_presentation` | ce notebook : vue d'ensemble |
| `01_orders` | nettoyage de `orders` |
| `02_order_items` | nettoyage de `order_items` |
| `03_order_payments` | nettoyage de `order_payments` |
| `04_products` | nettoyage de `products` (et traduction des catégories) |
| à venir | `customers`, `sellers`, `order_reviews`, `geolocation` |
| à venir | fusion des tables nettoyées |